# Task 4 - Decision and Planning

## Contribution Breakdown

Team Members:
- Lukas Kurz (K12007739)
- Shamekh Al-Suwi (K12146739)
- Tobias Washüttl (K11916576)
- Daniel Buchberger (K0885317)

_Note: We made use of AI assistance/LLMs for parts of the code to organize, clean up and help with documenting it for easier readability, as is standard practice nowadays. This does not in any way mean that code was plagiarized or copied, unless explicitly stated. All work was done in best conscience by the contributors named above._


## Behavioral Planning

### What are Finite State Machines (FSM)?

A Finite State Machine (FSM) is a computational model used to represent and control execution flow. It consists of a finite number of states, transitions between those states, and actions. In the context of autonomous vehicles, FSMs are used to model the decision-making process for different driving scenarios.

### States of our Behavioral Planning FSM

Our FSM consists of three main states:

1. **LANE FOLLOWING**: The default state where the vehicle follows the lane at nominal speed.
2. **DECELERATION STOP**: An intermediate state where the vehicle is decelerating to come to a complete stop.
3. **STOP**: The vehicle is completely stopped for a required duration (5 seconds) before proceeding.

### Transitions Between States

The transitions between states are governed by specific conditions:

1. **LANE FOLLOWING → DECELERATION STOP**: Triggered when an object or intersection requiring a stop is detected within the lookahead distance.
2. **DECELERATION STOP → STOP**: Occurs when the vehicle has decelerated and reached the stopping point.
3. **STOP → LANE FOLLOWING**: Happens after the vehicle has remained stopped for the required 5 seconds.

### FSM Diagram

```
    ┌─────────────────┐          ┌───────────────────┐         ┌─────────────┐
    │                 │  Object  │                   │ Reached │             │
    │ LANE FOLLOWING  ├─────────►│ DECELERATION STOP ├────────►│    STOP     │
    │                 │ Detected │                   │  Stop   │             │
    └─────────┬───────┘          └───────────────────┘         └──────┬──────┘
              ▲                                                       │
              │                                                       │
              └───────────────────────────────────────────────────────┘
                            After 5-second wait complete
```

### Implementation Details

- **Lookahead Distance**: We set an appropriate lookahead distance to detect objects and intersections ahead of time.
- **Goal Setting**: For deceleration, we set a goal slightly behind the stopping point to ensure proper stopping behavior.
- **Speed Control**: We implemented different speed targets for each state (nominal for LANE FOLLOWING, zero for STOP).
- **State Transitions**: We use distance-based conditions rather than speed-based conditions for more reliable transitions.

## Path and trajectory generation using cubic spirals

### Cubic Spirals

Cubic spirals are mathematical curves used for trajectory generation in autonomous vehicles. The key advantage of cubic spirals is that they provide smooth transitions in curvature, which translates to comfortable steering for passengers.

A cubic spiral can be defined parametrically, where the curvature κ(s) is a cubic function of the arc length s:

κ(s) = a₀ + a₁s + a₂s² + a₃s³

This curvature profile ensures:  
- Smooth transitions between straight and curved segments
- Feasible paths that respect the vehicle's kinematic constraints
- Paths that can connect arbitrary start and goal poses

### Necessity of Goal Offsets

Goal offsets are crucial in trajectory planning for several reasons:

1. **Lane Maneuverability**: Offsets allow the planner to generate multiple potential trajectories across the width of the lane.
2. **Obstacle Avoidance**: By creating paths with lateral offsets, the vehicle can navigate around obstacles while staying on the road.
3. **Path Diversity**: Generating multiple offset goals allows the planner to evaluate different options and select the optimal one.
4. **Robustness**: Having multiple candidate paths increases the likelihood of finding at least one viable path in complex environments.

### Goal Offset Calculations

Our implementation generates multiple goal candidates by applying lateral offsets to the main goal point:

1. First, we determine the number of paths (goals) to generate.
2. We calculate the maximum lateral offset based on the lane width.
3. We distribute the offsets evenly across the lane, generating multiple goal points.
4. For each offset, we create a new goal pose (x, y, heading) by shifting the original goal perpendicular to the path direction.

### Collision Checking

For each generated spiral path, we perform collision detection to ensure safety:

1. We discretize the spiral into a sequence of points.
2. For each point, we create a simplified vehicle footprint (rectangle or circle).
3. We check for intersections between this footprint and any detected obstacles.
4. If a collision is detected, the path is marked as invalid and assigned a high cost.

### Spiral Cost Function

We evaluate each candidate spiral using a weighted cost function that considers multiple factors:

1. **Path Length**: Shorter paths are generally preferred to minimize travel time.
2. **Curvature**: Paths with lower maximum curvature provide more comfortable rides.
3. **Curvature Derivative**: Smooth changes in curvature reduce passenger discomfort.
4. **Lateral Deviation**: Paths closer to the lane center are preferred for safety.
5. **Obstacle Clearance**: Paths with greater distance to obstacles are safer.

The final cost is a weighted sum of these individual costs, allowing us to balance safety, comfort, and efficiency.

## Velocity Profile Generation

### 1. Brief Explanation of Velocity Profile Generation

The **Velocity Profile Generator** computes a sequence of velocities (a velocity profile) for a vehicle along a given path (the "spiral") to execute various driving maneuvers safely and comfortably.  
The goal is to transition from the current speed to a target speed—either by accelerating, decelerating, or coming to a stop—depending on the driving scenario.

Depending on the maneuver, different profiles are generated:
- **Nominal/Lane Follow:** The vehicle accelerates or decelerates to a desired speed.
- **Decelerate to Stop:** The vehicle smoothly decelerates to a stop.
- **Follow Vehicle:** (Not yet implemented) The vehicle matches the speed of a lead vehicle.

### 2. How Are the Profiles Calculated?

#### a) General Workflow

1. **Maneuver Detection:** The generator selects the appropriate profile function based on the current driving state (e.g., stopping, following, lane keeping).
2. **Velocity Calculation:**  
   For each point along the trajectory, the target velocity is computed, based on the start speed, target speed, maximum allowed acceleration/deceleration, and the distance traveled.
3. **Time Calculation:** For each point, the relative time is also determined, resulting in a time-based trajectory.

#### b) Key Calculation Formulas

- **Distance with Constant Acceleration:**  
  \[ d = \frac{v_f^2 - v_i^2}{2a} \]  
  (Distance \(d\) to go from initial velocity \(v_i\) to final velocity \(v_f\) at constant acceleration \(a\))

- **Final Speed after a Distance:**  
  \[ v_f = \sqrt{v_i^2 + 2ad} \]  
  (Final speed \(v_f\) after distance \(d\) from initial speed \(v_i\) at acceleration \(a\))

### 3. How Are Distances or Velocities Calculated for Each Point?

#### a) Distance Calculation Between Points

- The distance between two consecutive points on the trajectory is computed using the helper function `path_point_distance()`.
- For many calculations (such as when to start braking), the cumulative distance along the trajectory is summed.

#### b) Velocity Calculation for Each Point

- **Acceleration/Deceleration Phase:**  
  For each segment, the new velocity is calculated using the above formula (\(v_f\)), based on the distance to the next point and the current velocity.
- **Constant Speed Phase:**  
  Once the target speed is reached, the velocity remains constant for the rest of the trajectory (unless further deceleration is needed).

#### c) Time Calculation

- The time between two points is calculated as:  
  \[ \Delta t = \frac{|v_{i+1} - v_i|}{a_\text{max}} \]  
  (during acceleration/deceleration)  
  or  
  \[ \Delta t = \frac{\text{distance}}{v} \]  
  (during constant speed)

### 4. Typical Workflow for Each Maneuver

#### a) Decelerate to Stop
- Calculate how much distance is needed to first decelerate to a "slow speed," then to a full stop.
- If the required braking distance exceeds the remaining path, the deceleration is adjusted so that the vehicle stops exactly at the end of the path.
- Otherwise, the profile consists of three phases: deceleration to slow speed, constant slow speed, and final braking to zero.

#### b) Nominal Trajectory (Lane Follow)
- Calculate the distance required to transition from the current speed to the target speed.
- Up to this point, the vehicle accelerates or decelerates; after that, it maintains the target speed.

### 5. Example Workflow (Pseudocode)

For each maneuver:
1. Determine start and target speed.
2. Calculate required distance for acceleration/deceleration phase.
3. Iterate over the trajectory, for each point:
    - Calculate distance to the next point.
    - Compute new velocity using the kinematic formula.
    - Compute time to the next point.
    - Add all values as a TrajectoryPoint to the trajectory.

### 6. Key Methods in the Code

- **`calc_distance(v_i, v_f, a)`**: Calculates the distance needed to go from \(v_i\) to \(v_f\) at acceleration \(a\).
- **`calc_final_speed(v_i, a, d)`**: Calculates the final speed after distance \(d\) from \(v_i\) at acceleration \(a\).
- **`decelerate_trajectory()`**: Generates a profile for smooth stopping.
- **`nominal_trajectory()`**: Generates a profile for reaching a target speed.
- **`generate_trajectory()`**: Selects the appropriate profile function based on the maneuver.

### 7. Summary

The Velocity Profile Generator creates a physically plausible velocity profile for each driving maneuver by applying basic kinematic equations and respecting comfort/safety constraints (e.g., max acceleration).  
The calculation is performed point-by-point along the planned route, providing for each point both the velocity and the corresponding time.

## Analysis

### System Integration and Performance

Our integrated planning system successfully handles various driving scenarios, including lane following, stopping at intersections, and navigating around obstacles. The three-layer approach (behavioral planning, path generation, and velocity profiling) provides a robust and flexible framework for autonomous navigation.

### Strengths of the Implementation

1. **Robust State Machine**: The FSM design clearly separates different driving behaviors, making the system more maintainable and easier to debug. The state transitions are well-defined and based on reliable distance metrics rather than potentially noisy speed measurements.

2. **Adaptable Path Planning**: The cubic spiral approach generates smooth paths that respect vehicle kinematic constraints. Multiple candidate paths with different lateral offsets provide flexibility in obstacle avoidance while maintaining comfort.

3. **Safety-First Design**: Comprehensive collision checking ensures that only safe paths are selected. The weighted cost function balances multiple objectives, prioritizing safety while considering efficiency and comfort.

4. **Smooth Velocity Profiles**: The two-phase velocity profile generation creates comfortable acceleration and deceleration patterns. State-specific velocity calculations ensure appropriate speed control in different scenarios.

### Challenges and Limitations

1. **Computational Complexity**: Generating and evaluating multiple spiral paths can be computationally intensive, potentially limiting real-time performance on less powerful hardware.

2. **Parameter Tuning**: The system requires careful tuning of parameters like lookahead distance, maximum lateral offsets, and cost weights. Finding the optimal balance between different objectives requires extensive testing.

3. **Edge Cases**: While the system handles common scenarios well, unusual edge cases (like extremely sharp turns or complex intersection geometries) might require additional handling.

4. **Reactivity vs. Planning Horizon**: There's an inherent trade-off between quick reaction to changing conditions and maintaining a stable plan over a longer horizon.

### Future Improvements

1. **Dynamic Parameter Adjustment**: Implementing adaptive parameter tuning based on driving context could improve performance across different environments.

2. **Prediction Integration**: Incorporating better prediction of other road users' behaviors would enhance planning in dynamic environments.

3. **Machine Learning Optimization**: Using learning-based approaches could help optimize cost functions and parameter selection based on real-world performance data.

4. **Extended State Machine**: Adding more specialized states for complex maneuvers like lane changes, overtaking, or unprotected turns would increase the system's capabilities.

### Conclusion

Our implementation successfully addresses the core requirements of behavioral planning, path generation, and velocity control for autonomous navigation. The modular design allows for incremental improvements and extensions as needed. Through careful integration of these components, we've created a planning system that balances safety, comfort, and efficiency in a wide range of driving scenarios.